In [8]:
from agents import AsyncOpenAI
from agents import (
    Agent, Runner, 
    #function_tool,
    OpenAIChatCompletionsModel
)
#from agents.tracing import set_tracking_disabled

In [9]:
local_model = OpenAIChatCompletionsModel(
    model = "minimax-m2.7:cloud",
    openai_client=AsyncOpenAI(
        api_key="ollama",
        base_url="http://localhost:11434/v1"
    )
)

In [10]:
agent = Agent(
    name="Greeter",                                    # Give it a name
    instructions="You are a friendly assistant. Be concise.",  # Its personality/role
    model=local_model
)
# Step 2: Run it (sync version — good for scripts)
query = "Hello! What can you do?"
result = await Runner.run(agent, query)

# # Step 3: Get the output
print(result.final_output)


OPENAI_API_KEY is not set, skipping trace export




Hi! I'm an AI assistant that can help you with a wide range of tasks, including:

- **Answering questions** – explaining concepts, answering factual queries
- **Writing & editing** – drafts, emails, code, documents
- **Brainstorming** – ideas, solutions, plans
- **Analysis** – summarizing text, comparing options, problem-solving
- **Programming** – coding help, debugging, explanations
- **Research** – explaining complex topics in clear terms
- **Language tasks** – translation, grammar, tone adjustments

What can I help you with today?


In [11]:
agent = Agent(name="Analyst", instructions="You analyze topics briefly.", model=local_model)

result = await Runner.run(agent, "Why is Python popular for AI?")

# The key properties of RunResult:
print("=" * 50)
print(f"Final Output:  {result.final_output[:100]}...")   # The text response
print(f"Last Agent:    {result.last_agent.name}")          # Which agent finished
print(f"New Items:     {len(result.new_items)}")           # Items generated during run
print(f"Raw Responses: {len(result.raw_responses)}")       # Raw LLM responses
print("=" * 50)

OPENAI_API_KEY is not set, skipping trace export


Final Output:  

**Python’s dominance in AI comes from a mix of practical and cultural reasons:**

1. **Easy to lea...
Last Agent:    Analyst
New Items:     1
Raw Responses: 1


In [12]:
agent = Agent(
    name="Tutor",
    instructions="You are a Python tutor. Be concise. Remember context from previous messages.",
    model=local_model
)

# Turn 1
result = await Runner.run(agent, "What is a list in Python?")
print(f"Turn 1: {result.final_output}\n")

# Turn 2 — carry conversation history forward
new_input = result.to_input_list() + [{"role": "user", "content": "How do I sort one?"}]
result = await Runner.run(agent, new_input)
print(f"Turn 2: {result.final_output}\n")

# Turn 3
new_input = result.to_input_list() + [{"role": "user", "content": "What about reverse sort?"}]
result = await Runner.run(agent, new_input)
print(f"Turn 3: {result.final_output}")

OPENAI_API_KEY is not set, skipping trace export


Turn 1: 

A **list** in Python is an ordered, mutable collection of items enclosed in square brackets `[]`.

**Key features:**
- Can hold mixed data types
- Items are indexed (0, 1, 2...)
- Mutable (you can add, remove, or change items)

**Examples:**
```python
fruits = ["apple", "banana", "cherry"]
numbers = [1, 2, 3, 4, 5]
mixed = [1, "hello", 3.14, True]

# Accessing items
fruits[0]  # "apple"

# Modifying
fruits.append("orange")  # adds item
fruits[1] = "blueberry"  # changes item
```

**Common operations:**
| Operation | Code | Description |
|-----------|------|-------------|
| Append | `list.append(x)` | Add to end |
| Insert | `list.insert(i, x)` | Insert at index |
| Remove | `list.remove(x)` | Remove first occurrence |
| Pop | `list.pop(i)` | Remove at index |
| Sort | `list.sort()` | Sort in place |
| Length | `len(list)` | Number of items |

Lists are one of Python's most commonly used data structures.



OPENAI_API_KEY is not set, skipping trace export


Turn 2: 

**Main methods:**

### 1. `sort()` - modifies the original list
```python
numbers = [3, 1, 4, 1, 5]
numbers.sort()
print(numbers)  # [1, 1, 3, 4, 5]

# Reverse sort
numbers.sort(reverse=True)
```

### 2. `sorted()` - returns a new list
```python
numbers = [3, 1, 4, 1, 5]
sorted_numbers = sorted(numbers)
print(sorted_numbers)  # [1, 1, 3, 4, 5]
# Original list unchanged
```

### 3. Custom sorting
```python
# Sort by length
fruits = ["banana", "apple", "cherry"]
fruits.sort(key=len)
# ["apple", "banana", "cherry"]

# Sort descending
fruits.sort(reverse=True)

# Sort tuples by second element
pairs = [(1, "one"), (3, "three"), (2, "two")]
pairs.sort(key=lambda x: x[1])
# [(3, "three"), (1, "one"), (2, "two")]
```

**Quick reference:**
| Method | Modifies original? | Returns sorted list? |
|--------|-------------------|---------------------|
| `list.sort()` | Yes | `None` |
| `sorted(list)` | No | Yes |



OPENAI_API_KEY is not set, skipping trace export


Turn 3: 

**Two approaches:**

### 1. `reverse=True` parameter
```python
numbers = [3, 1, 4, 1, 5]
numbers.sort(reverse=True)
print(numbers)  # [5, 4, 3, 1, 1]
```

### 2. `reverse()` method - reverses order, not sorts
```python
numbers = [5, 3, 1]
numbers.reverse()
print(numbers)  # [1, 3, 5]
```

**With `sorted()`:**
```python
numbers = [3, 1, 4, 1, 5]
desc = sorted(numbers, reverse=True)
```

**Key difference:**
- `reverse=True` sorts descending
- `reverse()` just flips the current order (no sorting)


In [ ]:
agent = Agent(name="Streamer", instructions="Explain briefly.", mode=local_model)

# Streaming — see tokens as they arrive
result = Runner.run_streamed(agent, "What is machine learning in 2 sentences?")

async for event in result.stream_events():
    if event.type == "raw_response_event" and hasattr(event.data, 'delta'):
        print(event.data.delta, end="", flush=True)

print(f"\n\n--- Final: {result.final_output}")

TypeError: Agent.__init__() got an unexpected keyword argument 'mode'

OPENAI_API_KEY is not set, skipping trace export
